# Healthcare Cost Analysis

**Python • PostgreSQL • Tableau Public**

This notebook checks the supplied workbook's preview extracts, joins patients, visits, diagnosis reference data, and costs, summarizes cost patterns, profiles high-cost patients, and exports a Tableau-ready CSV.

> **Important:** The workbook provided for this project contains preview extracts. Results below cover only records successfully matched across these extracts and are not full-population financial estimates.

## 1. Load libraries and workbook

Place the authorized hackathon workbook at `data/healthcare_dataset.xlsx` before running this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'healthcare_dataset.xlsx'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError('Place healthcare_dataset.xlsx in the repository data/ folder. See data/README.md.')

patients = pd.read_excel(DATA_PATH, sheet_name='patients (preview)', header=1)
visits = pd.read_excel(DATA_PATH, sheet_name='visits (preview)', header=1)
costs = pd.read_excel(DATA_PATH, sheet_name='costs (preview)', header=1)
diagnoses = pd.read_excel(DATA_PATH, sheet_name='diagnoses_ref', header=1)
visits['visit_date'] = pd.to_datetime(visits['visit_date'], errors='coerce')
print('Workbook:', DATA_PATH)
print('Rows — patients, visits, costs, diagnosis reference:', len(patients), len(visits), len(costs), len(diagnoses))

## 2. Data quality and relationship checks

In [ ]:
table_specs = {
    'patients': (patients, 'patient_id'),
    'visits': (visits, 'visit_id'),
    'costs': (costs, 'visit_id'),
    'diagnoses_ref': (diagnoses, 'icd_code'),
}
quality = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': len(df.columns),
     'null_cells': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()),
     'duplicate_keys': int(df[key].duplicated().sum())}
    for name, (df, key) in table_specs.items()
])
display(quality)

coverage = pd.Series({
    'visits_without_patient_match': int((~visits['patient_id'].isin(patients['patient_id'])).sum()),
    'visits_without_diagnosis_match': int((~visits['icd_code'].isin(diagnoses['icd_code'])).sum()),
    'visits_without_cost_match': int((~visits['visit_id'].isin(costs['visit_id'])).sum()),
}, name='count')
display(coverage.to_frame())

## 3. Build the visit-level analysis table

Join keys: `patient_id`, `icd_code`, and `visit_id`. The left joins keep visits even when preview extracts do not contain a matching cost record.

In [ ]:
analysis = (
    visits
    .merge(patients, on='patient_id', how='left', validate='m:1', suffixes=('', '_patient'))
    .merge(diagnoses, on='icd_code', how='left', validate='m:1', suffixes=('', '_diagnosis'))
    .merge(costs, on='visit_id', how='left', validate='1:1')
)
costed = analysis.dropna(subset=['total_cost']).copy()
print(f'Visits in extract: {len(analysis):,}')
print(f'Visits matched to cost: {costed.visit_id.nunique():,}')
print(f'Visits without cost: {analysis.total_cost.isna().sum():,}')
print(f'Matched cost coverage: {len(costed) / len(analysis):.1%}')
display(costed.head())

## 4. Executive metrics and monthly trend

In [ ]:
summary = pd.Series({
    'patients_in_costed_visits': costed['patient_id'].nunique(),
    'costed_visits': costed['visit_id'].nunique(),
    'total_cost_in_matched_preview': round(costed['total_cost'].sum(), 2),
    'average_cost_per_matched_visit': round(costed['total_cost'].mean(), 2),
    'first_visit_date': costed['visit_date'].min(),
    'last_visit_date': costed['visit_date'].max(),
}, name='value')
display(summary.to_frame())

monthly = (costed.assign(month=costed['visit_date'].dt.to_period('M').astype(str))
           .groupby('month', as_index=False)
           .agg(visit_count=('visit_id', 'nunique'), total_cost=('total_cost', 'sum')))
ax = monthly.plot(x='month', y='total_cost', marker='o', legend=False,
                  title='Monthly Total Healthcare Cost — Matched Preview', figsize=(10, 5))
ax.set_xlabel('Month'); ax.set_ylabel('Total cost in source units')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
display(monthly)

## 5. Cost by diagnosis category

In [ ]:
category_costs = (costed.groupby('category', dropna=False)
    .agg(visit_count=('visit_id', 'nunique'), average_cost=('total_cost', 'mean'), total_cost=('total_cost', 'sum'))
    .sort_values('total_cost', ascending=False).reset_index())
category_costs[['average_cost', 'total_cost']] = category_costs[['average_cost', 'total_cost']].round(2)
display(category_costs)
ax = category_costs.sort_values('total_cost').plot(kind='barh', x='category', y='total_cost', legend=False,
    title='Total Cost by Diagnosis Category — Matched Preview', figsize=(10, 6))
ax.set_xlabel('Total cost in source units'); ax.set_ylabel('Diagnosis category')
plt.tight_layout(); plt.show()

## 6. Patient-level cost concentration

In [ ]:
patient_costs = (costed.groupby('patient_id').agg(
    total_cost=('total_cost', 'sum'), visit_count=('visit_id', 'nunique'),
    average_cost_per_visit=('total_cost', 'mean'), diagnosis_count=('icd_code', 'nunique')))
chronic_counts = (costed.loc[costed['chronic'].astype(str).str.lower().isin(['true', '1', 'yes'])]
    .groupby('patient_id')['icd_code'].nunique().rename('chronic_condition_count'))
patient_costs = patient_costs.join(chronic_counts).fillna({'chronic_condition_count': 0})
patient_costs['chronic_condition_count'] = patient_costs['chronic_condition_count'].astype(int)
patient_attributes = patients.set_index('patient_id')[['age', 'gender', 'insurance_type', 'smoking_status', 'risk_score']]
patient_costs = patient_costs.join(patient_attributes)
threshold = patient_costs['total_cost'].quantile(0.80)
patient_costs['top_cost_cohort'] = patient_costs['total_cost'] >= threshold
top_cost_total_share = patient_costs.loc[patient_costs['top_cost_cohort'], 'total_cost'].sum() / patient_costs['total_cost'].sum()
print(f'80th percentile cost threshold: {threshold:,.2f} (source units)')
print(f'Cost share represented by top-cost 20% cohort: {top_cost_total_share:.1%}')
display(patient_costs.sort_values('total_cost', ascending=False).head(20).round(2))

## 7. Export a Tableau-ready dataset

The export contains only visits with a matched cost record. It is a preview-based extract and not the full source population.

In [ ]:
export_columns = [
    'visit_id', 'patient_id', 'visit_date', 'visit_type', 'provider',
    'diagnosis_name', 'category', 'severity', 'chronic', 'age', 'gender',
    'insurance_type', 'smoking_status', 'risk_score', 'total_cost', 'insurance_paid', 'patient_paid'
]
missing = [c for c in export_columns if c not in costed.columns]
if missing:
    raise KeyError(f'Missing expected columns: {missing}')
tableau_data = costed[export_columns].copy()
tableau_data['age_band'] = pd.cut(tableau_data['age'], bins=[17, 34, 54, 74, float('inf')],
    labels=['18-34', '35-54', '55-74', '75+'], include_lowest=True)
tableau_data['visit_date'] = tableau_data['visit_date'].dt.strftime('%Y-%m-%d')
output_path = OUTPUT_DIR / 'tableau_data.csv'
tableau_data.to_csv(output_path, index=False)
print(f'Exported {len(tableau_data):,} matched preview rows to {output_path}')